#  08 — Silver DLT: streets_silver (union of 4 Bronze sources) + quarantine

## Configuration

In [0]:
import dlt
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

CATALOG = spark.conf.get("pipeline.catalog", "vstone_catalog")
BRONZE = spark.conf.get("pipeline.bronze_schema", "bronze")
SILVER = spark.conf.get("pipeline.silver_schema", "silver")

SILVER_PROPS = {
    "quality": "silver",
    "delta.enableChangeDataFeed": "true",
    "pipelines.reset.allowed": "true",
}
QUARANTINE_PROPS = {
    "quality": "quarantine",
    "delta.enableChangeDataFeed": "true",
    "pipelines.reset.allowed": "true",
}


## Pandas UDF: value-level cleaning (NOT header renaming — UDFs operate

In [0]:
#    on column values, not column names; see standardize_headers() below
#    for the actual header-standardization utility) 

@F.pandas_udf(StringType())
def clean_text(s: pd.Series) -> pd.Series:
    """Strip whitespace only, preserve case. Used where exact-text matching
    matters downstream (e.g. street names must match telegram.csv's text)."""
    return s.astype(str).str.strip()

##Header standardization utility

In [0]:
def standardize_headers(df):
    renamed = {c: c.strip().lower().replace(" ", "_") for c in df.columns if c != c.strip().lower().replace(" ", "_")}
    for old, new in renamed.items():
        df = df.withColumnRenamed(old, new)
    if renamed:
        print(f"  standardize_headers: renamed {renamed}")
    return df

## DEDUPLICATION

In [0]:
def deduplicate(df, partition_cols):
    """
    Remove duplicate records using the supplied business grain.
    """
    return df.dropDuplicates(partition_cols)

## streets_silver: union of the 4 Bronze technique-tables

In [0]:
def _build_streets_stream():
    def _bronze_stream(table):
        return (spark.readStream
                .format("delta")
                .option("ignoreDeletes", "true")
                .option("ignoreChanges", "true")
                .table(f"{CATALOG}.{BRONZE}.{table}"))
    return (
        _bronze_stream("streets_csv_copyinto")
        .unionByName(_bronze_stream("streets_autoloader"), allowMissingColumns=True)
        .unionByName(_bronze_stream("streets_dlt"), allowMissingColumns=True)
        .unionByName(_bronze_stream("streets_xml"), allowMissingColumns=True)
    )

## Transformation

In [0]:
def _transform_streets(df):
    df = standardize_headers(df)
    return (df.select(
        F.expr("try_cast(street_id as int)").alias("street_id"),
        F.to_timestamp(F.col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'").alias("reading_ts"),
        F.expr("try_cast(noise as double)").alias("noise"),
        F.expr("try_cast(pollution as double)").alias("pollution"),
        F.expr("try_cast(light as double)").alias("light"),
        F.expr("try_cast(raining as double)").alias("raining"),  # unclipped — Day 5 applies the [0,100] rule
        F.col("load_dt").alias("bronze_load_dt"),
        clean_text(F.col("source")).alias("bronze_source"),
    )
    .withColumn("silver_load_dt", F.current_timestamp()))

## Data quality rules


In [0]:
_STREETS_VALID_FILTER = (
    F.col("street_id").isNotNull() &
    F.col("reading_ts").isNotNull() &
    F.col("noise").isNotNull() &
    F.col("pollution").isNotNull() &
    F.col("light").isNotNull() &
    F.col("raining").isNotNull()
)

## Silver Table

In [0]:
@dlt.table(
    name="streets_silver",
    comment="Silver Streaming — unified street sensor readings from 4 Bronze sources "
            "(copyinto/autoloader/dlt/xml). Deduplicated by (street_id, reading_ts). "
            "raining is cast but NOT yet clipped to [0,100] — that's a Day 5 business rule.",
    table_properties=SILVER_PROPS,
)
@dlt.expect("valid_street_id", "street_id IS NOT NULL")
@dlt.expect("valid_reading_ts", "reading_ts IS NOT NULL")
@dlt.expect("valid_noise", "noise IS NOT NULL")
@dlt.expect("valid_pollution", "pollution IS NOT NULL")
@dlt.expect("valid_light", "light IS NOT NULL")
@dlt.expect("valid_raining", "raining IS NOT NULL")

def streets_silver():
    df = _transform_streets(_build_streets_stream())
    df = deduplicate(df, ["street_id", "reading_ts"])
    return df.filter(_STREETS_VALID_FILTER)


##Quarantine Table

In [0]:
@dlt.table(
    name="streets_silver_quarantine",
    comment="Quarantine Streaming — street readings rejected from streets_silver. "
            "Reasons: MISSING_OR_MALFORMED_STREET_ID | UNPARSABLE_DATE.",
    table_properties=QUARANTINE_PROPS,
)
def streets_silver_quarantine():
    df = _transform_streets(_build_streets_stream())
    df = deduplicate(df, ["street_id", "reading_ts"])
    return (df
            .filter(~_STREETS_VALID_FILTER)
            .withColumn("quarantine_reason",
                        F.when(F.col("street_id").isNull(), "MISSING_OR_MALFORMED_STREET_ID")
                        .when(F.col("reading_ts").isNull(), "UNPARSABLE_DATE")
                        .when(F.col("noise").isNull(), "MISSING_OR_MALFORMED_NOISE")
                        .when(F.col("pollution").isNull(), "MISSING_OR_MALFORMED_POLLUTION")
                        .when(F.col("light").isNull(), "MISSING_OR_MALFORMED_LIGHT")
                        .when(F.col("raining").isNull(), "MISSING_OR_MALFORMED_RAINING")
                        .otherwise("DATA_QUALITY_ISSUE"))
            .withColumn("quarantine_dt", F.current_timestamp()))